In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path = "/content/drive/MyDrive/266-final-project/data"

print("Files in data folder:")
print(os.listdir(data_path))

Files in data folder:
['ROCStories_winter2017 - ROCStories_winter2017.csv', 'cloze_test_val__winter2018-cloze_test_ALL_val.csv', 'cloze_test_test__winter2018-cloze_test_ALL_test.csv']


In [ ]:
for root, dirs, files in os.walk(data_path):
    print("FOLDER:", root)
    print("FILES:", files)
    print("-" * 50)

FOLDER: /content/drive/MyDrive/266-final-project/data
FILES: ['ROCStories_winter2017 - ROCStories_winter2017.csv', 'cloze_test_val__winter2018-cloze_test_ALL_val.csv', 'cloze_test_test__winter2018-cloze_test_ALL_test.csv']
--------------------------------------------------


In [ ]:
roc_path = os.path.join(data_path, "ROCStories_winter2017 - ROCStories_winter2017.csv")
cloze_val_path = os.path.join(data_path, "cloze_test_val__winter2018-cloze_test_ALL_val.csv")

roc = pd.read_csv(roc_path)
cloze_val = pd.read_csv(cloze_val_path)

print("ROC shape:", roc.shape)
print("Cloze validation shape:", cloze_val.shape)

ROC shape: (52665, 7)
Cloze validation shape: (1571, 8)


In [ ]:
print("ROC columns:")
print(roc.columns.tolist())
print()

print("Cloze validation columns:")
print(cloze_val.columns.tolist())

ROC columns:
['storyid', 'storytitle', 'sentence1', 'sentence2', 'sentence3', 'sentence4', 'sentence5']

Cloze validation columns:
['InputStoryid', 'InputSentence1', 'InputSentence2', 'InputSentence3', 'InputSentence4', 'RandomFifthSentenceQuiz1', 'RandomFifthSentenceQuiz2', 'AnswerRightEnding']


In [ ]:
print("ROC preview:")
display(roc.head())

print("Cloze validation preview:")
display(cloze_val.head())

ROC preview:


,storyid,storytitle,sentence1,sentence2,sentence3,sentence4,sentence5
0,8bbe6d11-1e2e-413c-bf81-eaea05f4f1bd,David Drops the Weight,David noticed he had put on a lot of weight re...,He examined his habits to try and figure out t...,He realized he'd been eating too much fast foo...,He stopped going to burger places and started ...,"After a few weeks, he started to feel much bet..."
1,0beabab2-fb49-460e-a6e6-f35a202e3348,Frustration,Tom had a very short temper.,One day a guest made him very angry.,He punched a hole in the wall of his house.,Tom's guest became afraid and left quickly.,Tom sat on his couch filled with regret about ...
2,87da1a22-df0b-410c-b186-439700b70ba6,Marcus Buys Khakis,Marcus needed clothing for a business casual e...,All of his clothes were either too formal or t...,He decided to buy a pair of khakis.,The pair he bought fit him perfectly.,Marcus was happy to have the right clothes for...
3,2d16bcd6-692a-4fc0-8e7c-4a6f81d9efa9,Different Opinions,Bobby thought Bill should buy a trailer and ha...,Bill thought a truck would be better for what ...,Bobby pointed out two vehicles were much more ...,Bill was set in his ways with conventional thi...,He ended up buying the truck he wanted despite...
4,c71bb23b-7731-4233-8298-76ba6886cee1,Overcoming shortcomings,John was a pastor with a very bad memory.,He tried to memorize his sermons many days in ...,He decided to learn to sing to overcome his ha...,He then made all his sermons into music and sa...,His congregation was delighted and so was he.


Cloze validation preview:


,InputStoryid,InputSentence1,InputSentence2,InputSentence3,InputSentence4,RandomFifthSentenceQuiz1,RandomFifthSentenceQuiz2,AnswerRightEnding
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,Rick grew up in a troubled household.,"He never found good support in family, and tur...",It wasn't long before Rick got shot in a robbery.,The incident caused him to turn a new leaf.,He is happy now.,He joined a gang.,1
1,bff9f820-9605-4875-b9af-fe6f14d04256,Laverne needs to prepare something for her fri...,She decides to bake a batch of brownies.,She chooses a recipe and follows it closely.,Laverne tests one of the brownies to make sure...,The brownies are so delicious Laverne eats two...,Laverne doesn't go to her friend's party.,1
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,Sarah had been dreaming of visiting Europe for...,She had finally saved enough for the trip.,She landed in Spain and traveled east across t...,She didn't like how different everything was.,Sarah then decided to move to Europe.,Sarah decided that she preferred her home over...,2
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,Gina was worried the cookie dough in the tube ...,She was very happy to find she was wrong.,The cookies from the tube were as good as from...,Gina intended to only eat 2 cookies and save t...,Gina liked the cookies so much she ate them al...,Gina gave the cookies away at her church.,1
4,69ac9b05-b956-402f-9fff-1f926ef9176b,It was my final performance in marching band.,I was playing the snare drum in the band.,We played Thriller and Radar Love.,The performance was flawless.,I was very proud of my performance.,I was very ashamed of my performance.,1


In [ ]:
# Show examples where ending1 has 0 sentences
cloze_val[cloze_val['ending1_sent_count'] == 0][
    ['RandomFifthSentenceQuiz1']
].head()


# # Show examples where ending2 has 0 sentences
# cloze_val[cloze_val['ending2_sent_count'] == 0][
#     ['RandomFifthSentenceQuiz2']
# ].head()


,RandomFifthSentenceQuiz1
391,"He agreed,"
1131,Cole thought about the incident every day afte...
1387,Jerry attempted to run the marathon but quickl...
1391,"They framed the letters and kept them for years,"
1552,"Paige decided to play in the snow,"


In [ ]:
from collections import Counter

all_text = " ".join(
    roc['sentence1'] + " " +
    roc['sentence2'] + " " +
    roc['sentence3'] + " " +
    roc['sentence4'] + " " +
    roc['sentence5']
)

words = all_text.lower().split()
word_freq = Counter(words)

word_freq.most_common(20)

[('the', 110439),
 ('to', 88158),
 ('a', 74158),
 ('was', 63033),
 ('he', 60735),
 ('she', 49620),
 ('and', 45947),
 ('his', 34442),
 ('her', 33223),
 ('i', 26612),
 ('it', 26544),
 ('in', 25691),
 ('of', 24635),
 ('had', 23366),
 ('for', 22347),
 ('on', 17893),
 ('they', 17347),
 ('at', 13868),
 ('with', 13070),
 ('that', 11490)]

I looked at the most common words across all stories.
This helps understand the dataset’s language — but also shows that common words dominate, so raw counts aren’t very useful.




In [ ]:
# Combine first 4 sentences
cloze_val['context'] = (
    cloze_val['InputSentence1'] + " " +
    cloze_val['InputSentence2'] + " " +
    cloze_val['InputSentence3'] + " " +
    cloze_val['InputSentence4']
)

cloze_val[['context', 'RandomFifthSentenceQuiz1', 'RandomFifthSentenceQuiz2']].head()

,context,RandomFifthSentenceQuiz1,RandomFifthSentenceQuiz2
0,Rick grew up in a troubled household. He never...,He is happy now.,He joined a gang.
1,Laverne needs to prepare something for her fri...,The brownies are so delicious Laverne eats two...,Laverne doesn't go to her friend's party.
2,Sarah had been dreaming of visiting Europe for...,Sarah then decided to move to Europe.,Sarah decided that she preferred her home over...
3,Gina was worried the cookie dough in the tube ...,Gina liked the cookies so much she ate them al...,Gina gave the cookies away at her church.
4,It was my final performance in marching band....,I was very proud of my performance.,I was very ashamed of my performance.


I combined the first 4 sentences into a single “context” input.
This is important because it matches how the model will actually see the data: context + possible endings.


In [ ]:
import pandas as pd

rows = []

for _, row in cloze_val.iterrows():
    context = row['InputSentence1'] + " " + row['InputSentence2'] + " " + row['InputSentence3'] + " " + row['InputSentence4']

    # Ending 1
    rows.append({
        "text": context + " " + row['RandomFifthSentenceQuiz1'],
        "label": 1 if row['AnswerRightEnding'] == 1 else 0
    })

    # Ending 2
    rows.append({
        "text": context + " " + row['RandomFifthSentenceQuiz2'],
        "label": 1 if row['AnswerRightEnding'] == 2 else 0
    })

df_model = pd.DataFrame(rows)

print(df_model.head())
print(df_model['label'].value_counts())

                                                text  label
0  Rick grew up in a troubled household. He never...      1
1  Rick grew up in a troubled household. He never...      0
2  Laverne needs to prepare something for her fri...      1
3  Laverne needs to prepare something for her fri...      0
4  Sarah had been dreaming of visiting Europe for...      0
label
1    1571
0    1571
Name: count, dtype: int64


llm

In [ ]:
import os
from openai import OpenAI

#os.environ["OPENAI_API_KEY"] = "sk-"
client = OpenAI()

def generate_one(prompt, model="gpt-4o-mini"):
    response = client.responses.create(
        model=model,
        input=prompt
    )
    return response.output_text.strip()


In [ ]:
roc = roc.reset_index(drop=True).copy()

roc["context"] = (
    roc["sentence1"] + " " +
    roc["sentence2"] + " " +
    roc["sentence3"] + " " +
    roc["sentence4"]
)

roc["gold_ending"] = roc["sentence5"]


In [ ]:
start_idx = 4940
end_idx = 5060


In [ ]:
roc_subset = roc.iloc[start_idx:end_idx].copy()

print(f"Processing stories {start_idx} to {end_idx-1}")
print("Total:", len(roc_subset))


Processing stories 4940 to 5059
Total: 120


In [ ]:

from pathlib import Path

OUTPUT_DIR=Path("/content/drive/MyDrive/266-final-project/data/roc_bundles")

RESULTS_PATH = OUTPUT_DIR / f"roc_negatives_{start_idx}_{end_idx}.jsonl"
ERRORS_PATH = OUTPUT_DIR / f"roc_errors_{start_idx}_{end_idx}.jsonl"


In [ ]:
def load_processed_story_ids(path):
    processed = set()
    if path.exists():
        with open(path, "r") as f:
            for line in f:
                if line.strip():
                    processed.add(json.loads(line)["story_id"])
    return processed

processed_ids = load_processed_story_ids(RESULTS_PATH)
print("Already processed:", len(processed_ids))


Already processed: 120


In [ ]:
remaining_df = roc_subset.loc[
    ~roc_subset.index.isin(processed_ids),
    ["context", "gold_ending"]
].copy()

remaining_df["story_id"] = remaining_df.index

print("Remaining:", len(remaining_df))


Remaining: 0


In [ ]:
def build_temporal_prompt(context, gold_ending):
    return f"""
You are creating a single-sentence incorrect ending for a narrative reasoning dataset.

Write ONE ending that:
- uses the same main character(s), setting, and topic
- is fluent, natural, and realistic
- is clearly WRONG only because it creates a TEMPORAL inconsistency
- must violate the expected order, timing, or duration of events
- should make something happen too early, too late, or in the wrong sequence
- the contradiction must be about WHEN events happen, not WHAT the character chooses to do
- must stay close in topic to the correct ending
- must not introduce new named characters, new locations, or unrelated events
- must not mainly be a sentiment flip
- must not mainly be a causal contradiction
- must be exactly one sentence

Context:
{context}

Correct ending:
{gold_ending}

Return only the incorrect ending sentence.
""".strip()


def build_causal_prompt(context, gold_ending):
    return f"""
You are creating a single-sentence incorrect ending for a narrative reasoning dataset.

Write ONE ending that:
- uses the same main character(s), setting, and topic
- is fluent, natural, and realistic
- is clearly WRONG only because it creates a CAUSAL inconsistency
- must directly contradict what should logically happen because of the earlier events
- the outcome must clearly contradict the expected result of earlier actions, not just be an alternative choice
- should preserve the same scenario but produce an illogical consequence, decision, or outcome
- must stay close in topic to the correct ending
- must not introduce new named characters, new locations, or unrelated events
- must not mainly be a timing/order mistake
- must not mainly be just an emotion reversal
- must not be a reasonable alternative continuation
- must be exactly one sentence

Context:
{context}

Correct ending:
{gold_ending}

Return only the incorrect ending sentence.
""".strip()


def build_state_prompt(context, gold_ending):
    return f"""
You are creating a single-sentence incorrect ending for a narrative reasoning dataset.

Write ONE ending that:
- uses the same main character(s), setting, and topic
- is fluent, natural, and realistic
- is clearly WRONG only because it creates a STATE inconsistency
- must contradict the established world state, condition, possession, location, status, or fact in the story
- should make the ending inconsistent with what is already true by the end of the context
- must stay close in topic to the correct ending
- must not introduce new named characters, new locations, or unrelated events
- must not mainly be a timing/order mistake
- must not mainly be a causal contradiction
- must not mainly be just an emotion reversal
- must be exactly one sentence

Context:
{context}

Correct ending:
{gold_ending}

Return only the incorrect ending sentence.
""".strip()


def build_sentiment_prompt(context, gold_ending):
    return f"""
You are creating a single-sentence incorrect ending for a narrative reasoning dataset.

Write ONE ending that:
- uses the same main character(s), setting, and topic
- is fluent, natural, and realistic
- is clearly WRONG only because it creates a SENTIMENT contradiction
- must reverse or conflict with the emotional direction of the story
- should preserve topic and situation while making the emotional outcome clearly inconsistent
- must stay close in topic to the correct ending
- must not introduce new named characters, new locations, or unrelated events
- must not mainly be a timing/order mistake
- must not mainly be a causal contradiction
- must be exactly one sentence

Context:
{context}

Correct ending:
{gold_ending}

Return only the incorrect ending sentence.
""".strip()

In [ ]:
def generate_one(prompt):
    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )
    return response.output_text.strip()


In [ ]:
def append_jsonl(path, rows):
    with open(path, "a") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")


In [ ]:
import time#(already-executed)
import json

BATCH_SIZE = 25
SLEEP = 0.4

rows = remaining_df.to_dict("records")

for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i:i+BATCH_SIZE]

    batch_results = []
    batch_errors = []

    print(f"\nBatch {i//BATCH_SIZE + 1}")

    for row in batch:
        sid = row["story_id"]
        context = row["context"]
        gold = row["gold_ending"]

        try:
            temporal = generate_one(build_temporal_prompt(context, gold))
            time.sleep(SLEEP)

            causal = generate_one(build_causal_prompt(context, gold))
            time.sleep(SLEEP)

            state = generate_one(build_state_prompt(context, gold))
            time.sleep(SLEEP)

            sentiment = generate_one(build_sentiment_prompt(context, gold))
            time.sleep(SLEEP)

            batch_results.append({
                "story_id": int(sid),
                "context": context,
                "gold_ending": gold,
                "temporal_negative": temporal,
                "causal_negative": causal,
                "state_negative": state,
                "sentiment_negative": sentiment
            })

            print(f"✓ {sid}")

        except Exception as e:
            batch_errors.append({
                "story_id": int(sid),
                "error": str(e)
            })
            print(f"✗ {sid}")

    append_jsonl(RESULTS_PATH, batch_results)
    append_jsonl(ERRORS_PATH, batch_errors)

    print(f"Saved {len(batch_results)} results")


In [ ]:
def read_jsonl(path):
    data = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    return pd.DataFrame(data)

generated_df = read_jsonl(RESULTS_PATH)
print("Total generated:", len(generated_df))
generated_df.head()


Total generated: 120


,story_id,context,gold_ending,temporal_negative,causal_negative,state_negative,sentiment_negative
0,4940,Tim and Jill wanted to get their house painted...,But the color turned out being too bright and ...,But they decided to paint their house yellow b...,"But when the paint was applied, they found it ...",But the color ended up being a dark navy blue ...,But they were utterly disappointed by the dull...
1,4941,"Kim knew her neighbor, Todd, was good at fixin...",And he wrote a program to crash her laptop aga...,And he returned her laptop just in time for he...,And he returned her laptop with a new operatin...,"And he returned her laptop without any issues,...",And he removed all her personal files and retu...
2,4942,Cindy devoted her life to helping animals. So ...,But the bear pulled the branch in along with C...,But Cindy had already fallen into the ditch be...,"Instead, the bear expertly climbed the branch ...",But the bear easily climbed out of the ditch w...,But the bear gratefully climbed out and ran of...
3,4943,Troy took up farming. Everything was going wel...,He ended up losing that month's harvest to ins...,He lost last year's harvest to insect.,"Instead, his crops thrived abundantly, untouch...","Instead, he proudly displayed a bountiful harv...",Troy was thrilled to see his plants thrive and...
4,4944,There was a new popular game out. Guillermo th...,But he received a permanent ban.,But he received a permanent ban before he even...,But he was awarded a special trophy for his ou...,"But he became the top player in the game, earn...",But he was celebrated as the game's top player...


YOUR_OPENAI_API_KEY

In [ ]:
import pandas as pd
df1=pd.read_json("/content/drive/MyDrive/266-final-project/data/roc_bundles/roc_negatives_0_500.jsonl",lines=True)
df1


,story_id,context,gold_ending,temporal_negative,causal_negative,state_negative,sentiment_negative
0,0,David noticed he had put on a lot of weight re...,"After a few weeks, he started to feel much bet...","To his surprise, he felt worse immediately aft...","After a few weeks, he found himself gaining ev...","After a few weeks, he found himself craving bu...","After a few weeks, he felt more sluggish and d..."
1,1,Tom had a very short temper. One day a guest m...,Tom sat on his couch filled with regret about ...,Tom sat on his couch laughing about how he had...,Tom felt an overwhelming sense of joy and invi...,Tom smiled as he proudly showed off the new mu...,"Tom grinned with satisfaction, feeling proud o..."
2,2,Marcus needed clothing for a business casual e...,Marcus was happy to have the right clothes for...,Marcus was surprised to realize that the event...,Marcus was disappointed that the khakis he bou...,Marcus was disappointed that the khakis he bou...,Marcus was disappointed that he couldn't find ...
3,3,Bobby thought Bill should buy a trailer and ha...,He ended up buying the truck he wanted despite...,He decided to buy the trailer Bobby suggested ...,He decided to follow Bobby's advice and purcha...,He decided to ignore his own preference and pu...,He joyfully accepted Bobby's suggestion and bo...
4,4,John was a pastor with a very bad memory. He t...,His congregation was delighted and so was he.,John began singing his sermons only after his ...,"However, his congregation found his musical se...",His congregation was confused and often left t...,His congregation criticized his singing and he...
...,...,...,...,...,...,...,...
495,495,My cousin's younger sister came to a wedding w...,They explained a beard is a male cover.,They explained a beard is a term used for some...,They collectively decided that the man must be...,They confidently declared that the man was her...,"They criticized her choice of date, believing ..."
496,496,The family had finally picked out a Christmas ...,There was a squirrel that had been living in t...,"Suddenly, they decided to put the tree in the ...","To their surprise, the tree began to glow with...",There was a hidden stash of ornaments that the...,"To their surprise, they found a family of racc..."
497,497,The news warned that a strong hurricane was co...,Harold was lucky and his home suffered minor d...,Harold was relieved to find that he had alread...,"Despite the devastation around him, Harold's h...",Harold's home was completely destroyed by the ...,"Harold's home was completely destroyed, and he..."
498,498,Susie broke her ankle and went to the orthoped...,"Then, the doctor put a new pink cast on Susie'...","Then, after the doctor put a new pink cast on ...","Then, the doctor decided to let Susie walk out...","Then, the doctor decided to leave Susie's ankl...","Then, the doctor decided to remove Susie’s ank..."


In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/266-final-project/data/roc_bundles")

neg = pd.read_json(OUTPUT_DIR / "roc_negatives_0_500.jsonl", lines=True)
err = pd.read_json(OUTPUT_DIR / "roc_errors_0_500.jsonl", lines=True)

print("neg rows:", len(neg))
print("err rows:", len(err))


neg rows: 500
err rows: 0


In [ ]:
import os
import json
import time
import pandas as pd
from pathlib import Path
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
client = OpenAI()

# -----------------------------
# Step 1: Prepare Cloze val data
# -----------------------------
cloze_subset = cloze_val.copy().reset_index(drop=True)

cloze_subset["context"] = (
    cloze_subset["InputSentence1"] + " " +
    cloze_subset["InputSentence2"] + " " +
    cloze_subset["InputSentence3"] + " " +
    cloze_subset["InputSentence4"]
)

# gold ending = the correct one based on AnswerRightEnding
cloze_subset["gold_ending"] = cloze_subset.apply(
    lambda row: row["RandomFifthSentenceQuiz1"] if row["AnswerRightEnding"] == 1
                else row["RandomFifthSentenceQuiz2"],
    axis=1
)

cloze_subset["story_id"] = cloze_subset.index

print("Total Cloze val stories:", len(cloze_subset))

# -----------------------------
# Step 2: Output paths
# -----------------------------
OUTPUT_DIR = Path("/content/drive/MyDrive/266-final-project/data/cloze_bundles")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / "cloze_negatives.jsonl"
ERRORS_PATH = OUTPUT_DIR / "cloze_errors.jsonl"

# -----------------------------
# Step 3: Resume support
# -----------------------------
def load_processed_story_ids(path):
    processed = set()
    if path.exists():
        with open(path, "r") as f:
            for line in f:
                if line.strip():
                    processed.add(json.loads(line)["story_id"])
    return processed

processed_ids = load_processed_story_ids(RESULTS_PATH)
print("Already processed:", len(processed_ids))

remaining_df = cloze_subset[~cloze_subset["story_id"].isin(processed_ids)].copy()
print("Remaining:", len(remaining_df))

# -----------------------------
# Step 4: Prompt builders (same as ROC)
# -----------------------------
def build_temporal_prompt(context, gold_ending):
    return f"""
You are creating a single-sentence incorrect ending for a narrative reasoning dataset.

Write ONE ending that:
- uses the same main character(s), setting, and topic
- is fluent, natural, and realistic
- is clearly WRONG only because it creates a TEMPORAL inconsistency
- must violate the expected order, timing, or duration of events
- should make something happen too early, too late, or in the wrong sequence
- the contradiction must be about WHEN events happen, not WHAT the character chooses to do
- must stay close in topic to the correct ending
- must not introduce new named characters, new locations, or unrelated events
- must not mainly be a sentiment flip
- must not mainly be a causal contradiction
- must be exactly one sentence

Context:
{context}

Correct ending:
{gold_ending}

Return only the incorrect ending sentence.
""".strip()

def build_causal_prompt(context, gold_ending):
    return f"""
You are creating a single-sentence incorrect ending for a narrative reasoning dataset.

Write ONE ending that:
- uses the same main character(s), setting, and topic
- is fluent, natural, and realistic
- is clearly WRONG only because it creates a CAUSAL inconsistency
- must directly contradict what should logically happen because of the earlier events
- the outcome must clearly contradict the expected result of earlier actions, not just be an alternative choice
- should preserve the same scenario but produce an illogical consequence, decision, or outcome
- must stay close in topic to the correct ending
- must not introduce new named characters, new locations, or unrelated events
- must not mainly be a timing/order mistake
- must not mainly be just an emotion reversal
- must not be a reasonable alternative continuation
- must be exactly one sentence

Context:
{context}

Correct ending:
{gold_ending}

Return only the incorrect ending sentence.
""".strip()

def build_state_prompt(context, gold_ending):
    return f"""
You are creating a single-sentence incorrect ending for a narrative reasoning dataset.

Write ONE ending that:
- uses the same main character(s), setting, and topic
- is fluent, natural, and realistic
- is clearly WRONG only because it creates a STATE inconsistency
- must contradict the established world state, condition, possession, location, status, or fact in the story
- should make the ending inconsistent with what is already true by the end of the context
- must stay close in topic to the correct ending
- must not introduce new named characters, new locations, or unrelated events
- must not mainly be a timing/order mistake
- must not mainly be a causal contradiction
- must not mainly be just an emotion reversal
- must be exactly one sentence

Context:
{context}

Correct ending:
{gold_ending}

Return only the incorrect ending sentence.
""".strip()

def build_sentiment_prompt(context, gold_ending):
    return f"""
You are creating a single-sentence incorrect ending for a narrative reasoning dataset.

Write ONE ending that:
- uses the same main character(s), setting, and topic
- is fluent, natural, and realistic
- is clearly WRONG only because it creates a SENTIMENT contradiction
- must reverse or conflict with the emotional direction of the story
- should preserve topic and situation while making the emotional outcome clearly inconsistent
- must stay close in topic to the correct ending
- must not introduce new named characters, new locations, or unrelated events
- must not mainly be a timing/order mistake
- must not mainly be a causal contradiction
- must be exactly one sentence

Context:
{context}

Correct ending:
{gold_ending}

Return only the incorrect ending sentence.
""".strip()

# -----------------------------
# Step 5: Generation loop
# -----------------------------
def generate_one(prompt):
    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )
    return response.output_text.strip()

def append_jsonl(path, rows):
    with open(path, "a") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

BATCH_SIZE = 25
SLEEP = 0.4

rows = remaining_df.to_dict("records")

for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i:i+BATCH_SIZE]
    batch_results = []
    batch_errors = []

    print(f"\nBatch {i//BATCH_SIZE + 1}")

    for row in batch:
        sid = row["story_id"]
        context = row["context"]
        gold = row["gold_ending"]

        try:
            temporal  = generate_one(build_temporal_prompt(context, gold))
            time.sleep(SLEEP)
            causal    = generate_one(build_causal_prompt(context, gold))
            time.sleep(SLEEP)
            state     = generate_one(build_state_prompt(context, gold))
            time.sleep(SLEEP)
            sentiment = generate_one(build_sentiment_prompt(context, gold))
            time.sleep(SLEEP)

            batch_results.append({
                "story_id":          int(sid),
                "context":           context,
                "gold_ending":       gold,
                "temporal_negative": temporal,
                "causal_negative":   causal,
                "state_negative":    state,
                "sentiment_negative":sentiment
            })
            print(f"✓ {sid}")

        except Exception as e:
            batch_errors.append({"story_id": int(sid), "error": str(e)})
            print(f"✗ {sid}")

    append_jsonl(RESULTS_PATH, batch_results)
    append_jsonl(ERRORS_PATH, batch_errors)
    print(f"Saved {len(batch_results)} results")

# -----------------------------
# Step 6: Load and verify
# -----------------------------
def read_jsonl(path):
    data = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    return pd.DataFrame(data)

cloze_generated = read_jsonl(RESULTS_PATH)
print("Total generated:", len(cloze_generated))
cloze_generated.head()


Total Cloze val stories: 1571
Already processed: 0
Remaining: 1571

Batch 1
✓ 0
✓ 1
✓ 2
✓ 3
✓ 4
✓ 5
✓ 6
✓ 7
✓ 8
✓ 9
✓ 10
✓ 11
✓ 12
✓ 13
✓ 14
✓ 15
✓ 16
✓ 17
✓ 18
✓ 19
✓ 20
✓ 21
✓ 22
✓ 23
✓ 24
Saved 25 results

Batch 2
✓ 25
✓ 26
✓ 27
✓ 28
✓ 29
✓ 30
✓ 31
✓ 32
✓ 33
✓ 34
✓ 35
✓ 36
✓ 37
✓ 38
✓ 39
✓ 40
✓ 41
✓ 42
✓ 43
✓ 44
✓ 45
✓ 46
✓ 47
✓ 48
✓ 49
Saved 25 results

Batch 3
✓ 50
✓ 51
✓ 52
✓ 53
✓ 54
✓ 55
✓ 56
✓ 57
✓ 58
✓ 59
✓ 60
✓ 61
✓ 62
✓ 63
✓ 64
✓ 65
✓ 66
✓ 67
✓ 68
✓ 69
✓ 70
✓ 71
✓ 72
✓ 73
✓ 74
Saved 25 results

Batch 4
✓ 75
✓ 76
✓ 77
✓ 78
✓ 79
✓ 80
✓ 81
✓ 82
✓ 83
✓ 84
✓ 85
✓ 86
✓ 87
✓ 88
✓ 89
✓ 90
✓ 91
✓ 92
✓ 93
✓ 94
✓ 95
✓ 96
✓ 97
✓ 98
✓ 99
Saved 25 results

Batch 5
✓ 100
✓ 101
✓ 102
✓ 103
✓ 104
✓ 105
✓ 106
✓ 107
✓ 108
✓ 109
✓ 110
✓ 111
✓ 112
✓ 113
✓ 114
✓ 115
✓ 116
✓ 117
✓ 118
✓ 119
✓ 120
✓ 121
✓ 122
✓ 123
✓ 124
Saved 25 results

Batch 6
✓ 125
✓ 126
✓ 127
✓ 128
✓ 129
✓ 130
✓ 131
✓ 132
✓ 133
✓ 134
✓ 135
✓ 136
✓ 137
✓ 138
✓ 139
✓ 140
✓ 141
✓ 142
✓ 143
✓ 144
✓ 145
✓ 146
✓ 147
✓ 148
✓ 149
Save

,story_id,context,gold_ending,temporal_negative,causal_negative,state_negative,sentiment_negative
0,0,Rick grew up in a troubled household. He never...,He is happy now.,"Before the incident, Rick was already happy an...",He decided to join another gang to seek reveng...,He still feels deeply resentful and hopeless a...,He feels more lost than ever and often regrets...
1,1,Laverne needs to prepare something for her fri...,The brownies are so delicious Laverne eats two...,Laverne tests one of the brownies after she ha...,The brownies turn out so terrible that Laverne...,"The brownies are burnt, so Laverne decides to ...","The brownies taste terrible, so Laverne decide..."
2,2,Sarah had been dreaming of visiting Europe for...,Sarah decided that she preferred her home over...,Sarah decided to return home immediately after...,Sarah found herself so enamored with the vibra...,Sarah was thrilled to discover that Europe fel...,Sarah felt so inspired by her experiences that...
3,3,Gina was worried the cookie dough in the tube ...,Gina liked the cookies so much she ate them al...,Gina intended to only eat 2 cookies and saved ...,Gina successfully resisted the temptation and ...,Gina decided to throw the remaining cookies aw...,Gina was so disappointed by the cookies that s...
4,4,It was my final performance in marching band....,I was very proud of my performance.,I felt a wave of disappointment the moment we ...,I was disqualified from the competition for no...,I felt embarrassed by my performance.,I felt utterly embarrassed by my performance.
